# Preprocessing: Genre-Conditioned Story Generation

This notebook converts the raw genre-story dataset into training-ready examples for a conditional encoder-decoder Transformer.

**Pipeline:**
1. Load & clean raw text
2. Construct conditional pairs: `(genre + prompt) → story`
3. Encode genre labels
4. Tokenize with GPT-2 tokenizer (full text — no truncation)
5. Analyze token-length distributions (to inform future truncation experiments)
6. Stratified train / val / test split
7. Build PyTorch `Dataset` & `DataLoader`
8. Save all processed artifacts

**Design decision:** We keep full-length sequences here. A truncated variant can be created later for comparison — this notebook saves the untruncated version as the baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from sklearn.model_selection import train_test_split
from collections import Counter
from pathlib import Path
import json
import pickle
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("All imports loaded.")

## 1. Load Raw Dataset

In [ ]:
data_path = Path("../data/1k_stories_100_genre.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Place `1k_stories_100_genre.csv` in the `../data/` folder."
    )

df = pd.read_csv(data_path)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

## 2. Text Cleaning

Light-touch cleaning to preserve the author's voice while removing noise that hurts tokenization.

In [ ]:
import re

def clean_text(text: str) -> str:
    """Normalize whitespace and strip stray artefacts, but keep original wording."""
    text = text.strip()
    text = re.sub(r"\r\n", "\n", text)         # unify line endings
    text = re.sub(r"[ \t]+", " ", text)         # collapse horizontal whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)      # max two consecutive newlines
    return text

df["story_clean"] = df["story"].astype(str).apply(clean_text)
df["title_clean"] = df["title"].astype(str).apply(clean_text)
df["genre_clean"] = df["genre"].astype(str).str.strip()

print(f"Rows before dropping empty stories: {len(df)}")
df = df[df["story_clean"].str.len() > 0].reset_index(drop=True)
print(f"Rows after: {len(df)}")

df[["genre_clean", "title_clean", "story_clean"]].head(3)

## 3. Construct Conditional Generation Pairs

Per the proposal: **`(genre + prompt) → story`**

We use the **title** as the natural prompt (it's short, always present, and semantically meaningful). The genre is prepended as a special token `[GENRE: <label>]`.

| Field | Content |
|---|---|
| **encoder_input** | `[GENRE: Science Fiction] The Chronicles of the Cosmic Rift` |
| **decoder_target** | full story text |

In [ ]:
def build_encoder_input(row):
    """Format: [GENRE: <label>] <title>"""
    return f"[GENRE: {row['genre_clean']}] {row['title_clean']}"

df["encoder_input"] = df.apply(build_encoder_input, axis=1)
df["decoder_target"] = df["story_clean"]

print("Sample conditional pairs:\n")
for i in range(3):
    print(f"--- Example {i} ---")
    print(f"  INPUT : {df.loc[i, 'encoder_input']}")
    print(f"  TARGET: {df.loc[i, 'decoder_target'][:120]}...")
    print()

## 4. Genre Label Encoding

Map every unique genre string to an integer id (and back). This is needed for the genre-embedding variant described in the proposal.

In [ ]:
genres_sorted = sorted(df["genre_clean"].unique())
genre2id = {g: i for i, g in enumerate(genres_sorted)}
id2genre = {i: g for g, i in genre2id.items()}

df["genre_id"] = df["genre_clean"].map(genre2id)

print(f"Total unique genres: {len(genre2id)}")
print(f"\nFirst 10 genre mappings:")
for g, i in list(genre2id.items())[:10]:
    print(f"  {i:3d} → {g}")

print(f"\nGenre id column sample:\n{df[['genre_clean', 'genre_id']].head()}")

## 5. Tokenizer Setup

We use the **GPT-2 tokenizer** (BPE, 50257 vocab). Special tokens are added for our genre-conditioned format.

No truncation is applied — sequences are tokenized at their full length so we can compare full-text vs truncated training later.

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

SPECIAL_TOKENS = {
    "bos_token": "<|bos|>",
    "eos_token": "<|eos|>",
    "pad_token": "<|pad|>",
    "sep_token": "<|sep|>",
}
num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)

BOS_ID = tokenizer.bos_token_id
EOS_ID = tokenizer.eos_token_id
PAD_ID = tokenizer.pad_token_id
SEP_ID = tokenizer.sep_token_id

print(f"Vocab size after adding {num_added} special tokens: {len(tokenizer)}")
print(f"BOS={BOS_ID}  EOS={EOS_ID}  PAD={PAD_ID}  SEP={SEP_ID}")

## 6. Tokenize All Examples (Full Text — No Truncation)

Each example becomes:
- **encoder_ids**: `[BOS] + tokenize(encoder_input) + [EOS]`
- **decoder_ids**: `[BOS] + tokenize(story) + [EOS]`

No padding yet — variable-length lists are stored per sample. Padding happens dynamically in the DataLoader collate function.

In [ ]:
from tqdm import tqdm

encoder_ids_list = []
decoder_ids_list = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing"):
    enc_tokens = tokenizer.encode(row["encoder_input"], add_special_tokens=False)
    dec_tokens = tokenizer.encode(row["decoder_target"], add_special_tokens=False)

    encoder_ids_list.append([BOS_ID] + enc_tokens + [EOS_ID])
    decoder_ids_list.append([BOS_ID] + dec_tokens + [EOS_ID])

df["encoder_ids"] = encoder_ids_list
df["decoder_ids"] = decoder_ids_list

df["enc_len"] = df["encoder_ids"].apply(len)
df["dec_len"] = df["decoder_ids"].apply(len)

print(f"\nTokenization complete.")
print(f"Encoder lengths — min: {df['enc_len'].min()}, max: {df['enc_len'].max()}, "
      f"mean: {df['enc_len'].mean():.0f}, median: {df['enc_len'].median():.0f}")
print(f"Decoder lengths — min: {df['dec_len'].min()}, max: {df['dec_len'].max()}, "
      f"mean: {df['dec_len'].mean():.0f}, median: {df['dec_len'].median():.0f}")

## 7. Token Length Distribution Analysis

Understanding the distribution helps pick truncation thresholds later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["enc_len"], bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("Encoder Input — Token Lengths")
axes[0].set_xlabel("Tokens")
axes[0].set_ylabel("Count")
axes[0].axvline(df["enc_len"].median(), color="red", ls="--", label=f'median={df["enc_len"].median():.0f}')
axes[0].legend()

axes[1].hist(df["dec_len"], bins=40, color="darkorange", edgecolor="white")
axes[1].set_title("Decoder Target — Token Lengths")
axes[1].set_xlabel("Tokens")
axes[1].set_ylabel("Count")
axes[1].axvline(df["dec_len"].median(), color="red", ls="--", label=f'median={df["dec_len"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
percentiles = [50, 75, 90, 95, 99, 100]
enc_pcts = np.percentile(df["enc_len"], percentiles)
dec_pcts = np.percentile(df["dec_len"], percentiles)

pct_df = pd.DataFrame({
    "percentile": [f"p{p}" for p in percentiles],
    "encoder_tokens": enc_pcts.astype(int),
    "decoder_tokens": dec_pcts.astype(int),
})
print("Token length percentiles (useful for choosing truncation thresholds later):\n")
print(pct_df.to_string(index=False))

## 8. Train / Validation / Test Split

80 / 10 / 10 split, **stratified by genre** so every genre appears proportionally in each set.

With only ~10 stories per genre, some genres will have very few val/test samples — that's expected at this dataset size.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["genre_id"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["genre_id"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print(f"Train genres: {train_df['genre_id'].nunique()}  |  "
      f"Val genres: {val_df['genre_id'].nunique()}  |  "
      f"Test genres: {test_df['genre_id'].nunique()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, (name, split_df) in zip(axes, [("Train", train_df), ("Val", val_df), ("Test", test_df)]):
    counts = split_df["genre_clean"].value_counts()
    ax.barh(counts.index[:15], counts.values[:15], color="steelblue")
    ax.set_title(f"{name} — Top 15 Genres")
    ax.set_xlabel("Count")
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 9. PyTorch Dataset & DataLoader

The `Dataset` stores variable-length token id lists. The `collate_fn` dynamically pads each batch to the longest sequence in that batch (no global truncation).

An **attention mask** is created so the model ignores padding tokens.

In [ ]:
class StoryDataset(Dataset):
    """Holds pre-tokenized (encoder_ids, decoder_ids, genre_id) triples."""

    def __init__(self, dataframe):
        self.encoder_ids = dataframe["encoder_ids"].tolist()
        self.decoder_ids = dataframe["decoder_ids"].tolist()
        self.genre_ids = dataframe["genre_id"].tolist()

    def __len__(self):
        return len(self.encoder_ids)

    def __getitem__(self, idx):
        return {
            "encoder_ids": self.encoder_ids[idx],
            "decoder_ids": self.decoder_ids[idx],
            "genre_id": self.genre_ids[idx],
        }


def collate_fn(batch):
    """Pad each batch dynamically to its longest sequence."""
    enc_seqs = [torch.tensor(b["encoder_ids"], dtype=torch.long) for b in batch]
    dec_seqs = [torch.tensor(b["decoder_ids"], dtype=torch.long) for b in batch]
    genre_ids = torch.tensor([b["genre_id"] for b in batch], dtype=torch.long)

    enc_padded = torch.nn.utils.rnn.pad_sequence(enc_seqs, batch_first=True, padding_value=PAD_ID)
    dec_padded = torch.nn.utils.rnn.pad_sequence(dec_seqs, batch_first=True, padding_value=PAD_ID)

    enc_mask = (enc_padded != PAD_ID).long()
    dec_mask = (dec_padded != PAD_ID).long()

    return {
        "encoder_ids": enc_padded,
        "encoder_mask": enc_mask,
        "decoder_ids": dec_padded,
        "decoder_mask": dec_mask,
        "genre_ids": genre_ids,
    }


print("StoryDataset and collate_fn defined.")

In [ ]:
BATCH_SIZE = 8

train_ds = StoryDataset(train_df)
val_ds = StoryDataset(val_df)
test_ds = StoryDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}  |  Test batches: {len(test_loader)}")

## 10. Sanity Checks

Verify shapes, padding behaviour, and that we can decode back to readable text.

In [ ]:
batch = next(iter(train_loader))

print("Batch tensor shapes:")
for k, v in batch.items():
    print(f"  {k:15s} → {tuple(v.shape)}")

print(f"\n--- Decode first sample in batch ---")
enc_ids = batch["encoder_ids"][0]
dec_ids = batch["decoder_ids"][0]

non_pad_enc = enc_ids[batch["encoder_mask"][0].bool()]
non_pad_dec = dec_ids[batch["decoder_mask"][0].bool()]

print(f"  ENCODER ({len(non_pad_enc)} tokens): {tokenizer.decode(non_pad_enc)}")
print(f"  DECODER ({len(non_pad_dec)} tokens): {tokenizer.decode(non_pad_dec)[:200]}...")

print(f"\n  Genre id: {batch['genre_ids'][0].item()} → {id2genre[batch['genre_ids'][0].item()]}")

In [ ]:
pad_counts_enc = (batch["encoder_ids"] == PAD_ID).sum(dim=1)
pad_counts_dec = (batch["decoder_ids"] == PAD_ID).sum(dim=1)

print("Padding tokens per sample in this batch:")
print(f"  Encoder: {pad_counts_enc.tolist()}")
print(f"  Decoder: {pad_counts_dec.tolist()}")
print(f"\n  Max encoder len in batch: {batch['encoder_ids'].shape[1]}")
print(f"  Max decoder len in batch: {batch['decoder_ids'].shape[1]}")

## 11. Save Processed Artifacts

Save everything needed to reproduce training without re-running this notebook:
- Tokenizer (with special tokens)
- Genre mappings (`genre2id`, `id2genre`)
- Processed splits as pickle files (token id lists + genre ids)
- Config metadata (vocab size, special token ids, split sizes)

In [ ]:
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer.save_pretrained(str(OUT_DIR / "tokenizer"))
print(f"Tokenizer saved to {OUT_DIR / 'tokenizer'}")

with open(OUT_DIR / "genre2id.json", "w") as f:
    json.dump(genre2id, f, indent=2)
with open(OUT_DIR / "id2genre.json", "w") as f:
    json.dump({str(k): v for k, v in id2genre.items()}, f, indent=2)
print("Genre mappings saved.")

def save_split(split_df, name):
    data = {
        "encoder_ids": split_df["encoder_ids"].tolist(),
        "decoder_ids": split_df["decoder_ids"].tolist(),
        "genre_ids": split_df["genre_id"].tolist(),
        "encoder_inputs_text": split_df["encoder_input"].tolist(),
        "decoder_targets_text": split_df["decoder_target"].tolist(),
    }
    path = OUT_DIR / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  {name}: {len(split_df)} samples → {path}")

print("\nSaving splits:")
save_split(train_df, "train")
save_split(val_df, "val")
save_split(test_df, "test")

In [ ]:
config = {
    "vocab_size": len(tokenizer),
    "num_genres": len(genre2id),
    "special_tokens": {
        "bos_id": BOS_ID,
        "eos_id": EOS_ID,
        "pad_id": PAD_ID,
        "sep_id": SEP_ID,
    },
    "split_sizes": {
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df),
    },
    "truncation": "none",
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}

with open(OUT_DIR / "preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Preprocessing config saved:\n")
for k, v in config.items():
    print(f"  {k}: {v}")

## 12. Summary

| Item | Value |
|---|---|
| **Tokenizer** | GPT-2 BPE + 4 special tokens |
| **Encoder input format** | `[BOS] [GENRE: <label>] <title> [EOS]` |
| **Decoder target format** | `[BOS] <full story> [EOS]` |
| **Truncation** | None (full text preserved) |
| **Padding** | Dynamic per-batch in `collate_fn` |
| **Splits** | 80 / 10 / 10 stratified by genre |

**Saved to `data/processed/`:**
- `tokenizer/` — GPT-2 tokenizer with special tokens
- `genre2id.json`, `id2genre.json` — label mappings
- `train.pkl`, `val.pkl`, `test.pkl` — tokenized splits
- `preprocessing_config.json` — reproducibility metadata

**Next notebook:** `3. Baseline Transformer` — build and train the encoder-decoder model using these artifacts.